In [0]:
from pyspark.sql import functions as F

In [0]:
data = [
    ("A101", 250000, "originated", "2026-10-10 10:00:00"),
    ("A102", 180000, "denied", "2026-10-10 10:00:00"),
    ("A102", 180000, "denied", "2026-10-10 10:00:00"),
    ("A103", 300000, "originated", "2026-10-10 09:00:00"),
    ("A103", 325000, "originated", "2026-10-11 10:00:00"),
    ("A104", 200000, "denied", "2026-10-11 09:00:00"),
    ("A104", 220000, "originated", "2026-10-11 09:00:00")
]

columns = [
    "application_id",
    "loan_amount",
    "action_taken",
    "source_updated_at"
]

df = spark.createDataFrame(data, columns)

df = df.withColumn(
    "source_updated_at",
    F.to_timestamp("source_updated_at")
)

display(df)

In [0]:

duplicate_ids = (
    df.groupBy("application_id")
      .agg(F.count("*").alias("record_count"))
      .filter(F.col("record_count") > 1)
)

display(duplicate_ids)


In [0]:

exact_duplicates = (
    df.groupBy(*df.columns)
      .count()
      .filter(F.col("count") > 1)
)

display(exact_duplicates)

In [0]:

source_count = df.count()

distinct_applications = (
    df.select("application_id")
      .distinct()
      .count()
)

print("Source records:", source_count)
print("Distinct applications:", distinct_applications)

if source_count != distinct_applications:
    raise ValueError(
        "Silver publication blocked: "
        "Repeated application IDs require resolution."
    )
